# Phase 10 — The Final Figures and Tables

**The idea in one line.** Everything the dissertation needs to *show* — one consistently styled set
of publication-quality figures and one table of headline numbers — regenerated from the saved
artefacts of Phases 1–9 and written to a `results/` folder, ready to drop into the write-up.

**Prerequisite:** Phases 6–9 must have been run first (this notebook only *reads* their saved
files; if one is missing, the error will name the phase to go run).

**Figure → chapter map:**

| Figure | File | Dissertation home |
|---|---|---|
| F1 — Latent space map by sector | `f1_latent_map.png` | Methodology (VAE validation) |
| F2 — Cluster stability | `f2_cluster_stability.png` | Methodology (clustering) |
| F3 — Pair selection funnel | `f3_pair_funnel.png` | Methodology (cointegration) |
| F4 — Walk-forward equity, gross vs net | `f4_walkforward_equity.png` | Results (headline) |
| F5 — Benchmark head-to-head | `f5_benchmark.png` | Results (comparison) |
| F6 — Pair lifetimes | `f6_pair_lifetimes.png` | Results (persistence) |
| F7 — Turnover vs market stress | `f7_turnover_vol.png` | Results (persistence) |
| Headline numbers | `headline_numbers.csv` | Results (summary table) |

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from walkforward import perf

plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
                     'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
os.makedirs('results/figures', exist_ok=True)

def show_and_save(fig, name):
    fig.savefig(f'results/figures/{name}.png')
    plt.show()
    print(f'saved results/figures/{name}.png')

## F1–F3 — the methodology figures

The latent-space map (does the fingerprint space organise sensibly?), the cluster stability bars
(are the groups real?), and the selection funnel (how 670 candidates became the traded shortlist) —
the three pictures that carry the methodology chapter.

In [ ]:
proj = pd.read_csv('data/model_input/latent_projections.csv', index_col=0)
fig, ax = plt.subplots(figsize=(9, 7))
palette = plt.cm.tab20.colors
for i, s in enumerate(sorted(proj.sector.unique())):
    m = proj.sector == s
    ax.scatter(proj.umap_x[m], proj.umap_y[m], s=22, c=[palette[i % 20]], label=s, alpha=0.8)
ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.set_title('Frozen-VAE latent space (UMAP projection), coloured by GICS sector')
show_and_save(fig, 'f1_latent_map')

diag = pd.read_csv('data/model_input/cluster_diagnostics.csv')
fig, ax = plt.subplots(figsize=(8, 4))
d = diag.sort_values('stability', ascending=False)
ax.bar([f'c{c}' for c in d.cluster], d.stability,
       color=['seagreen' if s else 'indianred' for s in d.stable])
ax.axhline(0.5, color='k', ls='--', lw=0.8)
ax.set_ylim(0, 1.05); ax.set_ylabel('bootstrap stability')
ax.set_title('Per-cluster bootstrap stability (kept >= 0.5)')
show_and_save(fig, 'f2_cluster_stability')

pairs_all = pd.read_csv('data/model_input/pairs_all.csv')
g1 = pairs_all[pairs_all.eg_pvalue < 0.05]
g2 = g1[g1.half_life.between(5, 60)]
g3 = g2[g2.spread_std >= 0.01]
funnel = [('within-cluster', len(pairs_all)), ('EG p<0.05', len(g1)),
          ('half-life 5-60d', len(g2)), ('spread>=1%', len(g3))]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([f[0] for f in funnel], [f[1] for f in funnel], color='slateblue')
for i, (_, v) in enumerate(funnel):
    ax.text(i, v + 8, str(v), ha='center')
ax.set_title('Pair selection funnel (2015-2019 formation)')
show_and_save(fig, 'f3_pair_funnel')

## F4–F5 — the results figures

The headline walk-forward equity curve (gross and net of costs, with its drawdown underneath), and
the benchmark head-to-head that answers the research question.

In [ ]:
wf_ret = pd.read_csv('data/model_input/wf_returns.csv', index_col=0, parse_dates=True)['ret']
wf_net = pd.read_csv('data/model_input/wf_net_returns.csv', index_col=0, parse_dates=True)['ret']
_, eq_g = perf(wf_ret)
m_net, eq_n = perf(wf_net)

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
ax[0].plot(eq_g.index, eq_g.values, color='navy', label='gross')
ax[0].plot(eq_n.index, eq_n.values, color='darkorange', label='net of costs')
ax[0].axhline(1, color='grey', ls='--', lw=0.7); ax[0].legend()
ax[0].set_title('Walk-forward equity curve, frozen VAE, quarterly re-selection (2019-10 to 2026-03)')
dd = eq_n / eq_n.cummax() - 1
ax[1].fill_between(dd.index, dd.values, color='indianred')
ax[1].set_title('Drawdown (net)')
show_and_save(fig, 'f4_walkforward_equity')

bench = pd.read_csv('data/model_input/benchmark_returns.csv', index_col=0, parse_dates=True)
fig, ax = plt.subplots(figsize=(11, 6))
for col, colr in [('vae', 'navy'), ('garch', 'darkorange'), ('correlation', 'seagreen'), ('buyhold', 'grey')]:
    _, eq = perf(bench[col].dropna())
    ax.plot(eq.index, eq.values, label=col, color=colr, lw=1.5 if col == 'vae' else 1.0)
ax.axhline(1, color='k', lw=0.5); ax.legend()
ax.set_title('Net-of-costs head-to-head: VAE vs GARCH vs correlation vs buy-and-hold')
show_and_save(fig, 'f5_benchmark')

## F6–F7 — the persistence figures

How long the traded pairs lived across the 27 quarters, and whether the pair list churned harder
when markets were rough.

In [ ]:
wf_pairs = pd.read_csv('data/model_input/wf_pairs.csv')
wf_pairs['pair'] = ['|'.join(sorted([a, b])) for a, b in zip(wf_pairs.asset1, wf_pairs.asset2)]
W = sorted(wf_pairs.window.unique())
order = wf_pairs.groupby('pair').window.agg(['min', 'count']).sort_values(['min', 'count']).index
presence = pd.DataFrame(0, index=order, columns=W)
for _, r in wf_pairs.iterrows():
    presence.loc[r.pair, r.window] = 1
fig, ax = plt.subplots(figsize=(10, max(5, len(order) * 0.14)))
ax.imshow(presence.values, aspect='auto', cmap='Blues', interpolation='nearest')
ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=5)
ax.set_xticks(range(len(W))); ax.set_xticklabels(W, fontsize=7); ax.grid(False)
ax.set_xlabel('window'); ax.set_title('Pair lifetimes across the walk-forward (dark = traded)')
show_and_save(fig, 'f6_pair_lifetimes')

persist = pd.read_csv('data/model_input/persistence_summary.csv', index_col=0)
from scipy.stats import spearmanr
rho, pval = spearmanr(persist.prior_quarter_vol, persist.turnover)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(persist.prior_quarter_vol, persist.turnover, s=45, color='navy', alpha=0.7)
b, a = np.polyfit(persist.prior_quarter_vol, persist.turnover, 1)
xs = np.linspace(persist.prior_quarter_vol.min(), persist.prior_quarter_vol.max(), 20)
ax.plot(xs, a + b * xs, color='r')
ax.set_xlabel('prior-quarter realised vol (annualised)')
ax.set_ylabel('pair-list turnover')
ax.set_title(f'Pair turnover vs market stress (Spearman {rho:+.2f}, p={pval:.3f})')
show_and_save(fig, 'f7_turnover_vol')

## The headline-numbers table

One machine-readable table of the numbers the abstract, results chapter, and viva slides will quote,
assembled purely from saved artefacts (nothing recomputed, so it cannot drift from the notebooks
that produced it).

In [ ]:
hp = pd.read_json('models/vae/hparams.json', typ='series')
ari = pd.read_csv('data/model_input/cluster_stability.csv').iloc[0]
shortlist = pd.read_csv('data/model_input/pairs_shortlist.csv')
static = pd.read_csv('data/model_input/strategy_metrics.csv', index_col=0)
wf_m = pd.read_csv('data/model_input/wf_metrics.csv', index_col=0).iloc[:, 0]
cost = pd.read_csv('data/model_input/cost_summary.csv', index_col=0).iloc[:, 0]
bench_m = pd.read_csv('data/model_input/benchmark_metrics.csv', index_col=0)

headline = pd.Series({
    'vae_val_elbo': hp.best_val_loss,
    'bootstrap_ari_mean': ari.ari_mean,
    'shortlist_pairs': len(shortlist),
    'static_portfolio_sharpe': static.loc['sharpe', 'portfolio'],
    'walkforward_gross_sharpe': wf_m['sharpe'],
    'walkforward_net_sharpe': cost['net_sharpe'],
    'walkforward_net_cum_return': cost['net_cum_return'],
    'cost_bps_per_side': cost['cost_bps'],
    'garch_net_sharpe': bench_m.loc['GARCH', 'sharpe'],
    'correlation_net_sharpe': bench_m.loc['Correlation', 'sharpe'],
    'buyhold_net_sharpe': bench_m.loc['Buy&Hold', 'sharpe'],
})
headline.to_csv('results/headline_numbers.csv')
print(headline.round(4).to_string())
print()
print('Saved results/headline_numbers.csv')

## Summary

`results/figures/` now holds the seven dissertation figures (300 dpi, consistent styling) and
`results/headline_numbers.csv` the quotable numbers — all regenerated from saved artefacts with a
single run of this notebook, so re-running any earlier phase and then this one keeps every figure
and number in sync. The write-up can begin.